# Absorption filters example

This example designs **frequency-dependent absorption filters** for an FDN (Feedback Delay Network) reverb. Each filter approximates a target T60 (reverberation time) curve over frequency for a given delay line.

**What it does:**
- Defines target T60 values at a few frequencies for two delay lines (e.g. 20 ms and 40 ms).
- Uses `pyFDN.absorption_filters` to compute FIR filter coefficients that realize these targets.
- Uses `pyFDN.absorption_to_t60` to evaluate the achieved T60 vs frequency.
- Plots the filter impulse responses and the target vs approximated T60 curves.

**Setup:** `target_frequency` and `target_t60` define the desired T60 (seconds) at each frequency for each delay. `filter_order` is chosen from the desired filter length in samples (here 10 ms). The first figure shows the impulse responses of the two absorption filters; the second compares the target T60 curves (markers) with the approximation (lines).

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pyFDN

In [ ]:
fs = 48000
target_frequency = np.array([0, 500, 5000, 8000, fs / 2])
# Define two separate T60 curves, then merge into one array
t60_curve_1 = np.array([2, 2, 2, 1, 1])  # seconds
t60_curve_2 = np.array([2, 2, 1, 0.5, 0.5])  # seconds
target_t60 = np.column_stack([t60_curve_1, t60_curve_2])

delays_ms = [20, 40]  # ms
delays = pyFDN.ms_to_smp(delays_ms, fs)

filter_length = 10  # ms
filter_order = int(pyFDN.ms_to_smp(filter_length, fs))

coeffs = pyFDN.absorption_filters(target_frequency, target_t60, filter_order, delays, fs)
t60, t60_f = pyFDN.absorption_to_t60(coeffs, np.array(delays), 2**10, fs)

# Plot filters
plt.figure()
plt.plot(pyFDN.mulaw_encode(coeffs.T))
plt.legend(['Filter 1', 'Filter 2'])
plt.xlabel("Time [samples]")
plt.ylabel("Amplitude [mu-law]")
plt.grid()

# Plot target vs approximation
plt.figure()
plt.plot(target_frequency, target_t60, 'o-')
plt.plot(t60_f, t60)
plt.legend(['Target 1', 'Target 2', 'Approx 1', 'Approx 2'])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Reverberation time [s]")
plt.xlim([0, fs / 2])
plt.ylim(bottom=0)
plt.grid()
plt.show()